# 18. Making Attention Fast

From *Transformers from Scratch*.

In [ ]:
# On Colab there is nothing on disk, so this fetches the two or
# three files the chapter uses. If you cloned the repository it
# finds them where they already are and downloads nothing.
import os
import sys
import urllib.request

BASE = "https://raw.githubusercontent.com/joshfestus/Transformers-from-Scratch/main/code/"
PLACES = [".", "code", "../code"]

for name in ["ch18_fast.py"]:
    if not any(os.path.exists(os.path.join(where, name))
               for where in PLACES):
        urllib.request.urlretrieve(BASE + name, name)

for where in PLACES:
    if os.path.isdir(where) and where not in sys.path:
        sys.path.insert(0, where)

What attention costs, and the four ways round it.

Every position against every position is a square number of comparisons. This file measures that cost in memory and in time, then measures what the usual remedies actually save.

In [ ]:
import time

import numpy as np

rng = np.random.default_rng(0)

In [ ]:
def softmax(scores):
    shifted = scores - scores.max(axis=-1, keepdims=True)
    e = np.exp(shifted)
    return e / e.sum(axis=-1, keepdims=True)

Bytes held by the score matrix, for one layer and one sequence.

In [ ]:
def score_memory(length, heads=32, bytes_each=2):
    return heads * length * length * bytes_each

Wall clock for one head at this sequence length.

In [ ]:
def time_attention(length, width=64, repeats=3):
    q = rng.standard_normal((length, width))
    k = rng.standard_normal((length, width))
    v = rng.standard_normal((length, width))
    best = float("inf")
    for _ in range(repeats):
        start = time.perf_counter()
        softmax(q @ k.T / np.sqrt(width)) @ v
        best = min(best, time.perf_counter() - start)
    return best

Bytes of keys and values kept during generation.

In [ ]:
def cache_sizes(length, layers=32, heads=32, d_head=128, groups=None,
                bytes_each=2):
    kv_heads = heads if groups is None else groups
    return 2 * layers * kv_heads * length * d_head * bytes_each

Key and value rows recomputed if nothing is kept between steps.

In [ ]:
def work_without_cache(steps):
    return sum(range(1, steps + 1))

One new row per step, and the rest read back.

In [ ]:
def work_with_cache(steps):
    return steps

In [ ]:
def main():
    print("THE SCORE MATRIX, ONE LAYER, 32 HEADS, HALF PRECISION")
    print("  length     scores held")
    for length in [512, 2048, 8192, 32768]:
        mb = score_memory(length) / 1e6
        shown = f"{mb:,.0f} MB" if mb < 1000 else f"{mb / 1000:,.1f} GB"
        print(f"  {length:<10} {shown}")

    print("\nMEASURED TIME FOR ONE HEAD, ON THIS MACHINE")
    print("  length   seconds     times longer   times the length")
    base = time_attention(256)
    for length in [256, 512, 1024, 2048]:
        taken = time_attention(length)
        print(f"  {length:<8} {taken:<11.4f} {taken / base:<14.1f} "
              f"{length / 256:.0f}")
    print("  doubling the length roughly quadruples the work")

    print("\nGENERATING 1000 TOKENS, KEY AND VALUE ROWS COMPUTED")
    without = work_without_cache(1000)
    with_cache = work_with_cache(1000)
    print(f"  without a cache  {without:,}")
    print(f"  with a cache     {with_cache:,}")
    print(f"  saved            {without / with_cache:.0f} times the work")

    print("\nWHAT THE CACHE COSTS, 32 LAYERS, 8192 TOKENS")
    print("  scheme              key and value heads   cache")
    for name, groups in [("multi-head", None), ("grouped, 8", 8),
                         ("multi-query", 1)]:
        size = cache_sizes(8192, groups=groups)
        heads = 32 if groups is None else groups
        print(f"  {name:<19} {heads:<21} {size / 1e9:.2f} GB")
    print("  the queries stay at 32 heads in all three")

In [ ]:
main()

The chapter this came from works through every number above, and explains why each step is there.